# DevSquad V3.5.0-C 交互式教程

## Plan C: 分层架构 - CLI视图层 over 11阶段全生命周期

---

本Notebook将引导您了解DevSquad的统一生命周期架构，包括：
- ✅ 核心概念与架构
- 🔗 CLI命令到11阶段映射
- 🚧 Gate检查机制
- 📊 可视化与监控
- 🧪 实际应用示例

## 1. 环境准备与导入

In [ ]:
import sys
import os

# 添加项目路径
sys.path.insert(0, '..')

# 导入核心模块
from scripts.collaboration.lifecycle_protocol import (
    LifecycleMode,
    PhaseDefinition,
    ViewMapping,
    LifecycleProtocol,
    FullLifecycleAdapter,
    get_shared_protocol,
    VIEW_MAPPINGS,
    FULL_LIFECYCLE_PHASES,
)

print("✅ 环境准备完成")
print(f"📚 已加载 {len(FULL_LIFECYCLE_PHASES)} 个生命周期阶段")

## 2. 核心概念理解

In [ ]:
# 显示三种生命周期模式
print("🔄 DevSquad 支持三种生命周期模式：\n")

for mode in LifecycleMode:
    print(f"  • {mode.value}: {mode.name}")

print("\n📌 Plan C 使用 FULL 模式（完整11阶段）")

In [ ]:
# 查看11个生命周期阶段定义
print("📋 完整11阶段生命周期：\n")

for idx, phase in enumerate(FULL_LIFECYCLE_PHASES, 1):
    print(f"P{idx:02d} [{phase.phase_id}] {phase.name}")
    print(f"     角色: {phase.role_id}")
    print(f"     描述: {phase.description[:60]}...\n")

## 3. CLI命令映射 (Plan C核心)

In [ ]:
# 获取共享协议实例
protocol = get_shared_protocol()

print("🔗 CLI命令 → 11阶段映射关系：\n")

# 显示每个CLI命令映射到的阶段
for cmd_name, mapping in VIEW_MAPPINGS.items():
    # 解析命令到具体阶段
    phases = protocol.resolve_command_to_phases(cmd_name)
    phase_ids = [p.phase_id for p in phases] if phases else mapping.phases
    
    print(f"命令: {cmd_name.upper()}")
    print(f"  映射到阶段: {', '.join(phase_ids)}")
    print(f"  模式: {mapping.mode}")
    print(f"  Gate检查: {mapping.gate}\n")

In [ ]:
# 示例：'build'命令解析
cmd = 'build'
phases = protocol.resolve_command_to_phases(cmd)

print(f"🔍 命令 '{cmd}' 的详细解析：\n")

for phase in phases:
    print(f"阶段: {phase.phase_id} - {phase.name}")
    print(f"  负责角色: {phase.role_id}")
    print(f"  依赖阶段: {phase.dependencies if phase.dependencies else '无'}")
    print(f"  输入产物: {phase.artifacts_in[:40] if phase.artifacts_in else '无'}...")
    print(f"  输出产物: {phase.artifacts_out[:40] if phase.artifacts_out else '无'}...")
    print()

## 4. Gate检查机制

In [ ]:
# 测试各命令的Gate状态
test_commands = ['spec', 'plan', 'build', 'test', 'review', 'ship']

print("🚧 各命令Gate检查结果：\n")

for cmd in test_commands:
    result = protocol.check_command_gate(cmd)
    
    status_icon = "✅" if result.passed else "❌"
    print(f"{status_icon} {cmd.upper()}: {result.verdict}")
    
    if result.red_flags:
        print(f"   ⚠️  发现 {len(result.red_flags)} 个红旗")
    
    if result.missing_evidence:
        print(f"   📋 缺少 {len(result.missing_evidence)} 个证据")
    
    print()

In [ ]:
# 详细查看某个命令的Gate结果
cmd = 'build'
result = protocol.check_command_gate(cmd)

print(f"📊 '{cmd}' 命令详细Gate报告：\n")
print(f"通过状态: {'✅ 通过' if result.passed else '❌ 未通过'}")
print(f"判定结果: {result.verdict}")
print(f"红旗数量: {len(result.red_flags)}")
print(f"缺失证据: {len(result.missing_evidence)}")

if result.gap_report:
    print(f"\n差距报告:\n{result.gap_report[:300]}...")

## 5. 阶段执行模拟

In [ ]:
# 模拟执行一个完整的工作流
from scripts.collaboration.lifecycle_protocol import PhaseState

task = "实现用户认证功能"
command = 'build'

print(f"🎯 任务: {task}")
print(f"📝 命令: {command.upper()}\n")

# 解析需要执行的阶段
phases_to_execute = protocol.resolve_command_to_phases(command)

print("📋 执行计划：\n")
for i, phase in enumerate(phases_to_execute, 1):
    print(f"{i}. [{phase.phase_id}] {phase.name}")
    print(f"   执行者: {phase.role_id}")
    print()

In [ ]:
# 模拟执行并跟踪状态
import time

print("⏱️  开始模拟执行...\n")

for phase in phases_to_execute:
    print(f"▶️  执行阶段: {phase.phase_id} - {phase.name}")
    print(f"   角色: {phase.role_id}")
    
    # 模拟执行时间
    time.sleep(0.5)  # 模拟处理时间
    
    # 标记为已完成（模拟）
    try:
        protocol.mark_phase_completed(phase.phase_id)
        print(f"   ✅ 完成")
    except Exception as e:
        print(f"   ⚠️  {e}")
    
    print()

print("🎉 所有阶段执行完成！")

## 6. 状态查询与可视化

In [ ]:
# 查询当前系统状态
status = protocol.get_status()

print("📊 当前系统状态：\n")
print(f"当前模式: {status.get('mode', 'UNKNOWN')}")
print(f"当前阶段: {status.get('current_phase', 'N/A')}")
print(f"已完成阶段: {len(status.get('completed_phases', []))}")
print(f"运行中阶段: {len(status.get('running_phases', []))}")
print(f"失败阶段: {len(status.get('failed_phases', []))}")

In [ ]:
# 使用可视化工具显示进度
try:
    sys.path.insert(0, '../scripts/cli')
    from cli_visual import VisualFormatter, Colors, Icons
    
    vf = VisualFormatter(use_color=False)  # Notebook中禁用颜色
    
    # 显示生命周期头部信息
    vf.print_lifecycle_header(
        command='build',
        mapping=VIEW_MAPPINGS.get('build'),
        preset={'required_roles': ['architect', 'coder'], 'mode': 'full', 'gate': 'quality_gate'}
    )
    
    # 显示阶段列表
    all_phases = protocol.get_all_phases()
    vf.print_phase_list(all_phases[:8])  # 只显示前8个
    
except ImportError as e:
    print(f"⚠️  可视化模块未加载: {e}")

## 7. 性能基准测试

In [ ]:
# 运行性能测试
import time

print("🧪 性能基准测试\n")
print("=" * 50)

# 测试1: 命令解析性能
start = time.time()
iterations = 100

for _ in range(iterations):
    for cmd in ['spec', 'plan', 'build', 'test', 'review', 'ship']:
        protocol.resolve_command_to_phases(cmd)

elapsed = time.time() - start
avg_time = (elapsed / iterations / 6) * 1000  # 转换为毫秒

print(f"✅ 命令解析测试 ({iterations}次迭代):")
print(f"   总耗时: {elapsed:.3f}s")
print(f"   平均每次: {avg_time:.2f}ms")
print()

In [ ]:
# 测试2: Gate检查性能
start = time.time()

for _ in range(iterations):
    for cmd in ['spec', 'plan', 'build', 'test', 'review', 'ship']:
        protocol.check_command_gate(cmd)

elapsed = time.time() - start
avg_time = (elapsed / iterations / 6) * 1000

print(f"✅ Gate检查测试 ({iterations}次迭代):")
print(f"   总耗时: {elapsed:.3f}s")
print(f"   平均每次: {avg_time:.2f}ms")
print()

In [ ]:
# 测试3: 状态查询性能
start = time.time()

for _ in range(iterations * 10):
    protocol.get_status()

elapsed = time.time() - start
avg_time = (elapsed / (iterations * 10)) * 1000

print(f"✅ 状态查询测试 ({iterations * 10}次迭代):")
print(f"   总耗时: {elapsed:.3f}s")
print(f"   平均每次: {avg_time:.2f}ms")
print()
print("=" * 50)
print("🎉 所有性能测试通过！")

## 8. 实际应用场景

In [ ]:
# 场景1: 新功能开发工作流
print("📌 场景1: 开发新API端点\n")

task = "开发用户资料API"
workflow = ['spec', 'plan', 'build', 'test', 'review']

print(f"任务: {task}")
print(f"工作流: {' → '.join(workflow)}\n")

for step in workflow:
    phases = protocol.resolve_command_to_phases(step)
    phase_names = [p.name for p in phases]
    print(f"{step.upper()}:")
    print(f"  执行阶段: {', '.join(phase_names)}")
    
    gate = protocol.check_command_gate(step)
    gate_status = "✅ Pass" if gate.passed else "❌ Block"
    print(f"  Gate状态: {gate_status}\n")

In [ ]:
# 场景2: 快速修复流程
print("📌 场景2: 紧急Bug修复\n")

bug_task = "修复登录超时问题"
quick_workflow = ['plan', 'build', 'test', 'ship']

print(f"任务: {bug_task}")
print(f"快速工作流: {' → '.join(quick_workflow)}\n")

for step in quick_workflow:
    phases = protocol.resolve_command_to_phases(step)
    print(f"{step.upper()}: {[p.phase_id for p in phases]}\n")

## 9. 高级特性

In [ ]:
# 自定义工作流
print("🔧 创建自定义工作流\n")

custom_phases = ['P03', 'P04', 'P05', 'P06', 'P09']

print("选择阶段:")
for pid in custom_phases:
    phase = next((p for p in FULL_LIFECYCLE_PHASES if p.phase_id == pid), None)
    if phase:
        print(f"  ✓ {pid}: {phase.name}")

print(f"\n自定义工作流包含 {len(custom_phases)} 个阶段")

In [ ]:
# Checkpoint管理（跨会话状态持久化）
from scripts.collaboration.checkpoint_manager import CheckpointManager

checkpoint_mgr = CheckpointManager()

print("💾 Checkpoint状态管理：\n")
print(f"Checkpoint目录: {checkpoint_mgr.checkpoint_dir}")
print(f"支持的操作:")
print("  • save_checkpoint() - 保存当前状态")
print("  • load_checkpoint() - 加载已保存状态")
print("  • list_checkpoints() - 列出所有检查点")
print("  • clear_checkpoints() - 清除所有检查点")

## 10. 总结与下一步

In [ ]:
print("=" * 60)
print("🎓 DevSquad V3.5.0-C 教程总结")
print("=" * 60)
print()
print("✅ 已学习内容:")
print("  • 三种生命周期模式 (SHORTCUT/FULL/CUSTOM)")
print("  • 完整11阶段生命周期定义")
print("  • CLI命令到阶段的自动映射")
print("  • Gate检查与质量控制机制")
print("  • 阶段执行与状态跟踪")
print("  • 性能基准测试方法")
print("  • 实际应用场景示例")
print()
print("📚 推荐下一步:")
print("  1. 查看 docs/USAGE_GUIDE.md 了解完整用法")
print("  2. 运行 examples/quick_start.py 快速上手")
print("  3. 启动 Streamlit Dashboard: streamlit run scripts/dashboard.py")
print("  4. 阅读 docs/spec/SPEC_Lifecycle_Unified_Architecture_C.md")
print()
print("🚀 开始您的DevSquad之旅吧！")